# ACC Reinstatement vs Condition-Specific Subject-Level GC (Theta) — Cue-On Time Windows

This notebook mirrors the MATLAB subject-level reinstatement correlation workflow, correlating ACC reinstatement with condition-specific subject-level GC (`gc_net_z`) from **cue_on** epochs. Theta-band GC is averaged over the full cue window (0–1 s), the first half (0–0.5 s), or the second half (0.5–1 s) using time-resolved `merged_gc_rpe.csv` from the MVGC RPE pipeline.


In [ ]:
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr
from IPython.display import display

In [ ]:
# ---- Config ----
reinstatement_metric = 'MeanAccuracy'  # or 'ReinstateEffect'
corr_type = 'spearman'  # 'spearman' or 'pearson'
run_secondary_pearson = True
label_points = True
gc_metric = 'gc_net_z'
freq_band_label = 'theta'  # 4–8 Hz merged GC bundle below

acc_roi = 'ACC'
gc_roi_source = 'HPC'
gc_roi_target = 'ACC'

acc_reinstatement_csv = '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/crossdecoding/rpe_category/ACC_rpe_category_HFA_subject_reinstatement.csv'
# Time-resolved merged GC from cue_on run (theta 4–8 Hz); used to average gc_net_z within each cue sub-window
gc_merged_csv = (
    '/Volumes/T7 Shield/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mvgc_rpe/'
    'notebook_runs/cue_on_Day1_t0p0-1p0_sf50_4-8Hz_lag10_roie86d0eb437/merged_gc_rpe.csv'
)

# Seconds relative to cue_on (must lie within the cropped epoch, typically 0–1 s)
time_windows: Dict[str, Tuple[float, float]] = {
    'cue_full': (0.0, 1.0),
    'cue_first_half': (0.0, 0.5),
    'cue_second_half': (0.5, 1.0),
}

time_window_labels: Dict[str, str] = {
    'cue_full': 'whole cue (0–1 s)',
    'cue_first_half': 'first half (0–0.5 s)',
    'cue_second_half': 'second half (0.5–1 s)',
}

analysis_output_root = Path('/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/analysis_outputs/reinstatement_gc_rpe_theta')
figure_output_root = Path('/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-ACC_dmPFC_Reinstatement_GC_RPE')
analysis_output_root.mkdir(parents=True, exist_ok=True)
figure_output_root.mkdir(parents=True, exist_ok=True)

analysis_prefix = (
    f'rpe_category_HFA_{acc_roi}_{reinstatement_metric}_vs_{freq_band_label}_{gc_metric}_cue_on'
)
paths: Dict[str, object] = {
    'merged_csv': analysis_output_root / f'{analysis_prefix}_merged_subject_condition.csv',
    'results_csv': analysis_output_root / f'{analysis_prefix}_correlation_results.csv',
    'summary_txt': analysis_output_root / f'{analysis_prefix}_summary.txt',
    'acc_plot_pdfs': {
        wk: figure_output_root / f'{analysis_prefix}_ACC_grid_scatter_{wk}.pdf'
        for wk in time_windows
    },
    'sliding_corr_pdf': figure_output_root / f'{analysis_prefix}_sliding_spearman_gc_vs_reinstate.pdf',
    'sliding_corr_csv': analysis_output_root / f'{analysis_prefix}_sliding_spearman_gc_vs_reinstate.csv',
}

# Sliding-window Spearman: average gc_net_z over [center - W/2, center + W/2], then correlate across subjects
rpe_cats_sliding: Tuple[str, ...] = ('neg', 'neutral', 'pos')
sliding_window_len_s = 0.2   # seconds of GC averaged per window (matches binned analysis when aligned)
sliding_step_s = 0.01        # slide interval (s); use ~1/sample spacing for dense coverage
sliding_corr_min_n_subjects = 3

expected_conditions: List[Tuple[str, str]] = [
    ('cue', 'neg'),
    ('cue', 'neutral'),
    ('cue', 'pos'),
]

print('Analysis output root:', analysis_output_root)
print('Figure output root:', figure_output_root)
print('Analysis prefix:', analysis_prefix)
print('GC merged CSV:', gc_merged_csv)
print('Time windows (s):', time_windows)
print('Expected conditions:', expected_conditions)

print('Sliding RPE categories:', rpe_cats_sliding)
print('Sliding corr outputs:', paths['sliding_corr_pdf'])

print('Sliding window len (s):', sliding_window_len_s, '| step (s):', sliding_step_s)


In [ ]:
def assert_required_columns(tbl: pd.DataFrame, required_cols: List[str], table_name: str) -> None:
    missing = [c for c in required_cols if c not in tbl.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def deterministic_inner_join(left: pd.DataFrame, right: pd.DataFrame, keys: List[str]) -> pd.DataFrame:
    out = left.merge(right, on=keys, how='inner')
    return out.sort_values(keys).reset_index(drop=True)


def _extract_corr(res) -> Tuple[float, float]:
    if hasattr(res, 'statistic') and hasattr(res, 'pvalue'):
        return float(res.statistic), float(res.pvalue)
    return float(res[0]), float(res[1])


def run_correlation_analysis(
    x: np.ndarray,
    y: np.ndarray,
    corr_type: str = 'spearman',
    run_secondary_pearson: bool = True,
) -> Dict[str, float]:
    mask = np.isfinite(x) & np.isfinite(y)
    x_valid = x[mask]
    y_valid = y[mask]
    n = int(len(x_valid))
    if n < 3:
        raise ValueError(f'Need at least 3 valid observations; got {n}.')

    rho_s, p_s = _extract_corr(spearmanr(x_valid, y_valid, nan_policy='omit'))
    if run_secondary_pearson or corr_type.lower() == 'pearson':
        r_p, p_p = _extract_corr(pearsonr(x_valid, y_valid))
    else:
        r_p, p_p = np.nan, np.nan

    corr_type_norm = corr_type.lower()
    if corr_type_norm == 'spearman':
        primary_r, primary_p = rho_s, p_s
    elif corr_type_norm == 'pearson':
        primary_r, primary_p = r_p, p_p
    else:
        raise ValueError(f"Unsupported corr_type: {corr_type}. Use 'spearman' or 'pearson'.")

    return {
        'n': n,
        'spearman_rho': rho_s,
        'spearman_p': p_s,
        'pearson_r': r_p,
        'pearson_p': p_p,
        'primary_corr_type': corr_type_norm,
        'primary_r': primary_r,
        'primary_p': primary_p,
    }


def _fit_and_plot_line(ax, x: np.ndarray, y: np.ndarray) -> None:
    if len(x) < 2 or len(np.unique(x)) < 2:
        return
    coeff = np.polyfit(x, y, 1)
    xfit = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    yfit = np.polyval(coeff, xfit)
    ax.plot(xfit, yfit, color='#222222', linewidth=1.2)


def plot_roi_condition_grid(
    merged_df: pd.DataFrame,
    results_df: pd.DataFrame,
    target_col: str,
    target_label: str,
    gc_col: str,
    conditions: List[Tuple[str, str]],
    output_pdf: Path,
    time_window_key: str,
    time_window_label: str = '',
    label_points: bool = True,
) -> None:
    """Single-row grid over the requested condition panels."""
    n_panels = max(1, len(conditions))
    # Keep per-panel width consistent with prior 1x2 layout (12/2 = 6 each).
    fig_width = 6 * n_panels
    fig, axes = plt.subplots(1, n_panels, figsize=(fig_width, 5), constrained_layout=True)
    axes = np.atleast_1d(axes).flatten()

    sns.set_style('white')

    for ax, (task_condition, rpe_cat) in zip(axes, conditions):
        panel_df = merged_df[
            (merged_df['task_condition'] == task_condition)
            & (merged_df['rpe_cat'] == rpe_cat)
        ].copy()
        panel_df = panel_df[np.isfinite(panel_df[gc_col]) & np.isfinite(panel_df[target_col])]

        ax.set_title(f'{task_condition} / {rpe_cat}')
        if len(panel_df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            continue

        x = panel_df[gc_col].to_numpy(dtype=float)
        y = panel_df[target_col].to_numpy(dtype=float)

        sns.scatterplot(x=x, y=y, ax=ax, s=52, color='#4c78a8', edgecolor='black', linewidth=0.7)
        _fit_and_plot_line(ax, x, y)

        if label_points:
            for row in panel_df.itertuples(index=False):
                ax.text(getattr(row, gc_col), getattr(row, target_col), f' {row.Subject}', fontsize=8)

        res_match = results_df[
            (results_df['target_roi'] == target_label)
            & (results_df['time_window'] == time_window_key)
            & (results_df['task_condition'] == task_condition)
            & (results_df['rpe_cat'] == rpe_cat)
        ]
        if len(res_match) > 0:
            rr = res_match.iloc[0]
            ann = f"{rr['primary_corr_type']} r={rr['primary_r']:.3f}, p={rr['primary_p']:.3g}, n={int(rr['n'])}"
            ax.text(0.02, 0.98, ann, transform=ax.transAxes, ha='left', va='top', fontsize=10)

        ax.set_xlabel(gc_col)
        ax.set_ylabel(target_col)

        ax.grid(False)
        sns.despine(ax=ax, top=True, right=True)

    tw = time_window_label or time_window_key
    fig.suptitle(f'{target_label} reinstatement vs {gc_col} by condition ({tw})', fontsize=14)
    fig.savefig(output_pdf, dpi=300)
    # plt.close(fig)

In [ ]:
# ---- Load and assemble subject x condition table (per cue time window) ----
if not os.path.isfile(acc_reinstatement_csv):
    raise FileNotFoundError(f'ACC reinstatement CSV not found: {acc_reinstatement_csv}')
if not os.path.isfile(gc_merged_csv):
    raise FileNotFoundError(f'Merged GC CSV not found: {gc_merged_csv}')

acc_tbl = pd.read_csv(acc_reinstatement_csv)
gc_merged_df = pd.read_csv(gc_merged_csv)

assert_required_columns(acc_tbl, ['Subject', reinstatement_metric, 'NumElectrodes'], 'ACC reinstatement table')
assert_required_columns(
    gc_merged_df,
    ['participant', 'roi_source', 'roi_target', 'rpe_cat', 'time', gc_metric],
    'Merged GC table',
)

acc_metric_col = f'{acc_roi}_{reinstatement_metric}'

acc_tbl = acc_tbl.rename(
    columns={
        reinstatement_metric: acc_metric_col,
        'NumElectrodes': f'{acc_roi}_NumElectrodes',
    }
)[['Subject', acc_metric_col, f'{acc_roi}_NumElectrodes']]

reinstatement_tbl = acc_tbl.copy()


def average_gc_in_time_window(
    gc_df: pd.DataFrame,
    tmin: float,
    tmax: float,
    gc_roi_source: str,
    gc_roi_target: str,
    gc_metric: str,
) -> pd.DataFrame:
    """Average time-resolved gc_net_z within [tmin, tmax] for each participant x rpe_cat (cue_on only)."""
    m = (
        (gc_df['time'] >= tmin)
        & (gc_df['time'] <= tmax)
        & (gc_df['roi_source'] == gc_roi_source)
        & (gc_df['roi_target'] == gc_roi_target)
    )
    sub = gc_df.loc[m].copy()
    if len(sub) == 0:
        raise ValueError(
            f'No GC rows for {gc_roi_source}->{gc_roi_target} in time [{tmin}, {tmax}].'
        )
    out = (
        sub.groupby(['participant', 'rpe_cat'], as_index=False, dropna=False)
        .agg(**{gc_metric: (gc_metric, 'mean')})
    )
    out = out.rename(columns={'participant': 'Subject'})
    out['task_condition'] = 'cue'
    return out


window_tables: Dict[str, pd.DataFrame] = {}
for window_key, (tmin, tmax) in time_windows.items():
    gc_subject_condition = average_gc_in_time_window(
        gc_merged_df, tmin, tmax, gc_roi_source, gc_roi_target, gc_metric,
    )
    merged_w = deterministic_inner_join(gc_subject_condition, reinstatement_tbl, keys=['Subject'])
    if len(merged_w) == 0:
        raise ValueError(f'No overlapping subjects after merging GC and reinstatement for window {window_key}.')
    window_tables[window_key] = merged_w

merged_analysis_tbl = pd.concat(
    [df.assign(time_window=w) for w, df in window_tables.items()],
    ignore_index=True,
)

observed_conditions = set(
    tuple(x)
    for x in merged_analysis_tbl[['task_condition', 'rpe_cat']].drop_duplicates().itertuples(index=False, name=None)
)
missing_conditions = set(expected_conditions) - observed_conditions
if missing_conditions:
    raise ValueError(f'Missing expected condition pairs: {sorted(missing_conditions)}')

condition_counts = (
    merged_analysis_tbl
    .groupby(['time_window', 'task_condition', 'rpe_cat'], as_index=False)['Subject']
    .nunique()
    .rename(columns={'Subject': 'n_subjects'})
    .sort_values(['time_window', 'task_condition', 'rpe_cat'])
    .reset_index(drop=True)
)
if (condition_counts['n_subjects'] < 3).any():
    bad = condition_counts[condition_counts['n_subjects'] < 3]
    raise ValueError(f'Found condition(s) with <3 subjects: {bad.to_dict(orient="records")}')

missing_audit = merged_analysis_tbl[[
    'Subject', 'time_window', 'task_condition', 'rpe_cat', gc_metric, acc_metric_col,
    f'{acc_roi}_NumElectrodes',
]].isna().sum()

print('Subjects in ACC reinstatement table:', acc_tbl['Subject'].nunique())
print('Merged GC rows (all times):', len(gc_merged_df))
for wk, tbl in window_tables.items():
    print(f'  Window {wk}: merged subjects =', tbl['Subject'].nunique())
print('Rows in long merged table:', len(merged_analysis_tbl))
print('\nCondition counts (by time window):')
display(condition_counts)
print('\nMissing-value audit:')
display(missing_audit.to_frame(name='n_missing').T)
print('\nMerged table preview:')
display(merged_analysis_tbl.head())


In [ ]:
# ---- Sliding-window Spearman: GC vs ACC reinstatement (per RPE category) ----
# For each window center: average gc_net_z over [center - window_len/2, center + window_len/2]
# per subject and rpe_cat, then Spearman(GC_avg, reinstatement) across subjects.
# Also record mean GC and SEM (across subjects) per window for plotting vs rho.


def build_sliding_spearman_table(
    gc_long: pd.DataFrame,
    reinstate_tbl: pd.DataFrame,
    acc_metric_col: str,
    gc_roi_source: str,
    gc_roi_target: str,
    gc_metric: str,
    rpe_cats: Tuple[str, ...],
    window_len_s: float,
    step_s: float,
    min_n: int,
) -> pd.DataFrame:
    # ROI filter only — same as average_gc_in_time_window: mean all rows in [tmin,tmax]
    # per participant × rpe_cat (no per-time hemisphere collapse first).
    g = gc_long.loc[
        (gc_long['roi_source'] == gc_roi_source)
        & (gc_long['roi_target'] == gc_roi_target)
    ].copy()

    times = np.sort(g['time'].unique())
    t_lo = float(np.min(times))
    t_hi = float(np.max(times))
    half = window_len_s / 2.0
    if t_hi - t_lo < window_len_s:
        raise ValueError(
            f'Epoch [{t_lo}, {t_hi}] s is shorter than sliding_window_len_s={window_len_s}.'
        )
    start_c = t_lo + half
    end_c = t_hi - half
    centers = np.arange(start_c, end_c + step_s * 0.5, step_s, dtype=float)
    centers = centers[centers <= end_c + 1e-9]

    r = reinstate_tbl[['Subject', acc_metric_col]].copy()
    rows = []

    for center in centers:
        tmin = float(center - half)
        tmax = float(center + half)
        m = (g['time'] >= tmin) & (g['time'] <= tmax)
        sub_w = g.loc[m].copy()
        if len(sub_w) == 0:
            continue

        for cat in rpe_cats:
            sc = sub_w[sub_w['rpe_cat'] == cat].copy()
            if len(sc) == 0:
                rows.append({
                    'time': float(center),
                    'window_tmin': tmin,
                    'window_tmax': tmax,
                    'rpe_cat': cat,
                    'spearman_rho': np.nan,
                    'spearman_p': np.nan,
                    'se_rho': np.nan,
                    'mean_gc': np.nan,
                    'sem_mean_gc': np.nan,
                    'n_subjects': 0,
                })
                continue

            grp = (
                sc.groupby('participant', as_index=False, dropna=False)
                .agg(**{gc_metric: (gc_metric, 'mean')})
            )
            grp = grp.rename(columns={'participant': 'Subject'})
            merged = grp.merge(r, on='Subject', how='inner')
            x = merged[gc_metric].to_numpy(dtype=float)
            y = merged[acc_metric_col].to_numpy(dtype=float)
            mask = np.isfinite(x) & np.isfinite(y)
            x = x[mask]
            y = y[mask]
            n = int(len(x))
            mean_gc = float(np.nanmean(x)) if n > 0 else np.nan
            if n >= 2:
                sem_mean_gc = float(np.std(x, ddof=1) / np.sqrt(n))
            else:
                sem_mean_gc = np.nan
            if n < min_n:
                rows.append({
                    'time': float(center),
                    'window_tmin': tmin,
                    'window_tmax': tmax,
                    'rpe_cat': cat,
                    'spearman_rho': np.nan,
                    'spearman_p': np.nan,
                    'se_rho': np.nan,
                    'mean_gc': mean_gc,
                    'sem_mean_gc': sem_mean_gc,
                    'n_subjects': n,
                })
                continue
            rho, pval = spearmanr(x, y)
            rows.append({
                'time': float(center),
                'window_tmin': tmin,
                'window_tmax': tmax,
                'rpe_cat': cat,
                'spearman_rho': float(rho),
                'spearman_p': float(pval),
                'se_rho': np.nan,
                'mean_gc': mean_gc,
                'sem_mean_gc': sem_mean_gc,
                'n_subjects': n,
            })
    return pd.DataFrame(rows)


sliding_corr_df = build_sliding_spearman_table(
    gc_merged_df,
    reinstatement_tbl,
    acc_metric_col,
    gc_roi_source,
    gc_roi_target,
    gc_metric,
    rpe_cats_sliding,
    sliding_window_len_s,
    sliding_step_s,
    sliding_corr_min_n_subjects,
)
display(sliding_corr_df.head(20))
print('Sliding correlation rows:', len(sliding_corr_df))
print('Window centers:', len(sliding_corr_df['time'].unique()), 'unique')



In [ ]:
# ---- Plot sliding-window: Spearman rho (left) and mean GC ± SEM (right), one panel per RPE ----
rpe_order = list(rpe_cats_sliding)
color_map = {'neg': '#d62728', 'neutral': '#7f7f7f', 'pos': '#2ca02c'}
rpe_titles = {'neg': 'Negative RPE', 'neutral': 'Neutral RPE', 'pos': 'Positive RPE'}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), constrained_layout=True, sharex=True, sharey=True)
ax2_ref = None
for j, cat in enumerate(rpe_order):
    ax = axes[j]
    c = color_map.get(cat, f'C{j % 10}')
    sub = sliding_corr_df[sliding_corr_df['rpe_cat'] == cat].sort_values('time')
    t = sub['time'].to_numpy(dtype=float)
    rho = sub['spearman_rho'].to_numpy(dtype=float)
    mgc = sub['mean_gc'].to_numpy(dtype=float)
    sem_gc = sub['sem_mean_gc'].to_numpy(dtype=float)

    ax.plot(t, rho, color=c, linewidth=1.8, label='Spearman rho')
    ax.axhline(0.0, color='#bbbbbb', linewidth=1.0, linestyle='--')
    if j == 0:
        ax.set_ylabel('Spearman rho (GC vs reinstatement)')
    ax.tick_params(axis='y', labelleft=(j == 0))
    ax.set_xlabel('Window center time (s)')
    ax.set_title(rpe_titles.get(cat, cat))

    ax2 = ax.twinx()
    if ax2_ref is None:
        ax2_ref = ax2
    else:
        ax2.sharey(ax2_ref)
    m = np.isfinite(mgc) & np.isfinite(sem_gc)
    if m.any():
        lo = np.where(m, mgc - sem_gc, np.nan)
        hi = np.where(m, mgc + sem_gc, np.nan)
        ax2.fill_between(t, lo, hi, color=c, alpha=0.2, linewidth=0)
    ax2.plot(t, mgc, color=c, linewidth=1.4, linestyle='--', alpha=0.75, label=f'Mean {gc_metric} ± SEM')
    if j == len(rpe_order) - 1:
        ax2.set_ylabel(f'Mean {gc_metric} (across subjects)', color='#333333')
    ax2.tick_params(axis='y', labelcolor=c, labelright=(j == len(rpe_order) - 1))

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='best', frameon=False, fontsize=8)
    sns.despine(ax=ax, right=False)
    sns.despine(ax=ax2, left=True)

fig.suptitle(
    f'{gc_roi_source}→{gc_roi_target} {freq_band_label} vs {acc_metric_col} (cue_on) | '
    f'window {sliding_window_len_s:g} s, step {sliding_step_s:g} s | '
    f'GC band: SEM of mean across subjects',
    fontsize=11,
)
fig.savefig(paths['sliding_corr_pdf'], dpi=300)
sliding_corr_df.to_csv(paths['sliding_corr_csv'], index=False)
print('Saved:', paths['sliding_corr_pdf'])
print('Saved:', paths['sliding_corr_csv'])


In [ ]:
import seaborn as sns

ax = sns.barplot(
    data=merged_analysis_tbl,
    x='rpe_cat',
    y=gc_metric,
    hue='time_window',
    hue_order=list(time_windows.keys()),
)
ax.legend(title='Cue time window', bbox_to_anchor=(1.02, 1), loc='upper left')


In [ ]:
# ---- Run condition-specific correlations (ACC only; 3 time windows x 2 RPE levels) ----
result_rows = []
target_roi = 'ACC'
target_col = acc_metric_col

for time_window_key in time_windows:
    tbl = window_tables[time_window_key]
    for task_condition, rpe_cat in expected_conditions:
        sub = tbl[
            (tbl['task_condition'] == task_condition)
            & (tbl['rpe_cat'] == rpe_cat)
        ].copy()

        stats = run_correlation_analysis(
            x=sub[gc_metric].to_numpy(dtype=float),
            y=sub[target_col].to_numpy(dtype=float),
            corr_type=corr_type,
            run_secondary_pearson=run_secondary_pearson,
        )

        row = {
            'time_window': time_window_key,
            'target_roi': target_roi,
            'task_condition': task_condition,
            'rpe_cat': rpe_cat,
            'n': stats['n'],
            'spearman_rho': stats['spearman_rho'],
            'spearman_p': stats['spearman_p'],
            'pearson_r': stats['pearson_r'],
            'pearson_p': stats['pearson_p'],
            'primary_corr_type': stats['primary_corr_type'],
            'primary_r': stats['primary_r'],
            'primary_p': stats['primary_p'],
            'metric_name': reinstatement_metric,
            'gc_metric': gc_metric,
        }
        result_rows.append(row)

results_df = pd.DataFrame(result_rows)
results_df = results_df.sort_values(['time_window', 'target_roi', 'task_condition', 'rpe_cat']).reset_index(drop=True)

print('Correlation results (uncorrected p-values):')
display(results_df)


In [ ]:
# ---- Save merged table, correlation table, and text summary ----
merged_analysis_tbl.to_csv(paths['merged_csv'], index=False)
results_df.to_csv(paths['results_csv'], index=False)

summary_lines = []
summary_lines.append('ACC REINSTATEMENT VS CONDITION-SPECIFIC GC (THETA, CUE-ON TIME WINDOWS)')
summary_lines.append('====================================================================')
summary_lines.append('')
summary_lines.append(f'Reinstatement metric: {reinstatement_metric}')
summary_lines.append(f'Frequency band: {freq_band_label}')
summary_lines.append(f'GC metric: {gc_metric}')
summary_lines.append(f'GC pair: {gc_roi_source} -> {gc_roi_target}')
summary_lines.append(f'Time windows (s): {time_windows}')
summary_lines.append(f'Correlation type: {corr_type} (uncorrected p-values)')
summary_lines.append(f'Run secondary Pearson: {run_secondary_pearson}')
summary_lines.append(f'ACC reinstatement file: {acc_reinstatement_csv}')
summary_lines.append(f'Merged GC (time-resolved) file: {gc_merged_csv}')
summary_lines.append('')
summary_lines.append(f'Subjects in merged table: {merged_analysis_tbl["Subject"].nunique()}')
summary_lines.append('')
summary_lines.append('Condition counts (by time window):')
for row in condition_counts.itertuples(index=False):
    summary_lines.append(
        f'  {row.time_window}: {row.task_condition}/{row.rpe_cat}: n_subjects={row.n_subjects}'
    )
summary_lines.append('')
summary_lines.append('Correlation results:')
for row in results_df.itertuples(index=False):
    summary_lines.append(
        f'  window={row.time_window}, target={row.target_roi}, condition={row.task_condition}/{row.rpe_cat}, '
        f'n={row.n}, spearman_rho={row.spearman_rho:.6f}, spearman_p={row.spearman_p:.6g}, '
        f'pearson_r={row.pearson_r:.6f}, pearson_p={row.pearson_p:.6g}'
    )

with open(paths['summary_txt'], 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines) + '\n')

print('Saved merged analysis table:', paths['merged_csv'])
print('Saved correlation results:', paths['results_csv'])
print('Saved summary text:', paths['summary_txt'])


In [ ]:
# ---- Save cue scatter grids per cue time window (ACC only) ----
for wk, pdf_path in paths['acc_plot_pdfs'].items():
    plot_roi_condition_grid(
        merged_df=window_tables[wk],
        results_df=results_df,
        target_col=acc_metric_col,
        target_label='ACC',
        gc_col=gc_metric,
        conditions=expected_conditions,
        output_pdf=pdf_path,
        time_window_key=wk,
        time_window_label=time_window_labels.get(wk, wk),
        label_points=label_points,
    )
    print('Saved ACC grid plot:', pdf_path)